<a href="https://www.kaggle.com/code/jayhawk1900/f1-realmlp?scriptVersionId=321419938" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# F1 Pit Stop Prediction: RealMLP (Neural Network for Tabular Data)

**Competition:** [Playground Series S6E5 — Predicting F1 Pit Stops](https://www.kaggle.com/competitions/playground-series-s6e5)

**Metric:** ROC-AUC | **Task:** Binary classification — will this driver pit on the next lap?

---

## Why a Neural Network?

Gradient-boosted decision trees (XGBoost, CatBoost, LightGBM) usually dominate tabular competitions, and a tuned GBDT blend is a strong baseline. But blending models from the *same family* hits a ceiling — they make correlated errors.

This notebook implements **RealMLP**, a neural architecture (from the [PyTabKit](https://github.com/dholzmueller/pytabkit) research library) engineered specifically to make MLPs competitive with GBDTs on tabular data. It does this through three key ideas:

1. **PBLD embeddings** — encode numeric features as periodic (sine/cosine) waves at learned frequencies, letting the network create sharp, threshold-like responses similar to tree splits.
2. **Internal ensemble** — train 16 networks in parallel inside one model and average them, reducing variance.
3. **A carefully scheduled training recipe** — per-parameter-group learning rates, label-smoothing and dropout schedules, and class weighting for imbalance.

Because a neural network "sees" the data completely differently from trees, its predictions are **uncorrelated** with GBDTs — making it an ideal blend partner. A single RealMLP can match or beat an entire GBDT ensemble, and blending the two beats either alone.

---

## Approach

- Feature engineering: arithmetic interactions, numeric-as-categorical, count encoding, binning
- The original F1 strategy dataset concatenated as additional training data (a strong signal on this synthetic-but-ORIG-derived data)
- Cross-validated target encoding on key categorical interactions
- 5-fold stratified CV, GPU training (~25 min)

# F1 Pit Stop Prediction — RealMLP (Neural Network for Tabular Data)

**Competition:** [Playground Series S6E5 — Predicting F1 Pit Stops](https://www.kaggle.com/competitions/playground-series-s6e5)
**Metric:** ROC-AUC | **Task:** Binary classification — will this driver pit on the next lap?

---

## Overview

This notebook implements **RealMLP**, a neural-network architecture (from the [PyTabKit](https://github.com/dholzmueller/pytabkit) research library) engineered to make MLPs competitive with gradient-boosted trees on tabular data. Tree ensembles (XGBoost, CatBoost, LightGBM) usually dominate tabular problems, but blending models from the *same* family hits a ceiling. A neural network "sees" the data differently, so it makes uncorrelated errors — a single RealMLP can match or beat an entire GBDT ensemble, and the two families blend well together.

The full pipeline runs end to end in one self-contained cell. This version trains for **8 epochs** (an increase from an initial 4-epoch run, which was visibly undertrained — every fold's best epoch was its last).

---

## Pipeline, Start to Finish

**1. Setup & data loading.** Imports, seeding for reproducibility, GPU detection. Loads the competition train/test plus the original F1 strategy dataset (ORIG), which contains real lap data with the pit-stop label.

**2. Feature engineering.** A single function (fit on train, applied to test and ORIG) creates:
- *Arithmetic interactions* — ratios and products like `LapNumber/RaceProgress` and `LapTime × Cumulative_Degradation`
- *Numeric-as-categorical* — floored numeric values treated as categories, so embeddings can learn per-bucket effects
- *Count encoding* — each category replaced by its frequency
- *Quantile binning* — continuous features discretized into bins
- *Interaction categories* — `Race+Compound` and `Race+Year` combined for target encoding

**3. Model components.** Built from scratch in PyTorch:
- **NumericalPreprocessor** — median-centering, robust IQR scaling, smooth clipping
- **PBLDEmbedding** — *Periodic Basis with Learned Decay*: encodes each number through cosine waves at learned frequencies, letting the network form sharp, threshold-like responses similar to tree splits. This is the key idea that makes the MLP competitive.
- **CategoricalFeatureLayer** — one-hot for low-cardinality categories, learned embeddings for high-cardinality ones (e.g. Driver)
- **ScalingLayer** — a learned per-feature scale before the MLP
- **NTPLinear** — a linear layer carrying an `n_ens` dimension so 16 ensemble members compute in parallel via a single `einsum`
- **RealMLP** — assembles the above into a network with hidden layers `[512, 64, 128]`, an internal **16-member ensemble** averaged at inference

**4. Training recipe.** A carefully scheduled regime:
- *Per-parameter-group learning rates* (embeddings, scaling layer, first layer, other weights, biases each get their own)
- *Schedules* over training progress — learning rate (`flat_cos`), label smoothing (cosine decay from 0.04), dropout (exponential decay)
- *Class weighting* via BCE `pos_weight` to handle the ~80/20 imbalance
- *Gradient clipping* for stability

**5. K-fold training.** 5-fold stratified CV. In each fold:
- The **ORIG dataset is concatenated into the training portion** (validation stays pure competition data) — a strong signal on this synthetic-but-ORIG-derived data
- **Cross-validated target encoding** is fit inside the fold on the interaction categories (prevents leakage)
- A RealMLP is trained; out-of-fold and averaged test predictions are collected

**6. Outputs.** Final out-of-fold AUC, saved OOF/test prediction arrays (for later blending with GBDT models), and a submission file.

---

## Why It Works

Periodic embeddings + an internal ensemble + a tuned training recipe give a neural network performance that rivals gradient-boosted trees — while making fundamentally different errors. That diversity is what lets a neural model exceed a tree ensemble on its own, and blend with trees to exceed either alone.

In [1]:
# ============================================================
# RealMLP — complete single-cell pipeline (epochs=8)
# ============================================================
import os, math, random, time, warnings
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder
import torch
import torch.nn as nn
warnings.filterwarnings('ignore')

def seed_everything(seed):
    np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)

# ---------- Load data ----------
DATA_DIR = '/kaggle/input/competitions/playground-series-s6e5'
ORIG_DIR = '/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
orig = pd.read_csv(f'{ORIG_DIR}/f1_strategy_dataset_v4.csv')
print('Loaded:', train.shape, test.shape, orig.shape)

ID, TARGET = 'id', 'PitNextLap'
orig = orig.drop(['Normalized_TyreLife'], axis=1)
y_orig = orig[TARGET]; orig = orig.drop([TARGET], axis=1)
X = train.drop([ID, TARGET], axis=1); train_id = train[ID]
y = train[TARGET]
X_test = test.drop([ID], axis=1); test_id = test[ID]

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()

# ---------- Feature engineering ----------
category_map = {}
important_combos = [('Race', 'Compound'), ('Race', 'Year')]

def feature_engineering(df, fit=False):
    df = df.copy()
    df['_LapNumber_/_RaceProgress'] = (df['LapNumber'] / (df['RaceProgress'] + 1e-6)).astype('float32')
    df['_TyreLife_/_LapNumber'] = (df['TyreLife'] / df['LapNumber'].clip(lower=1)).astype('float32')
    df['_LapTime (s)_*_Cumulative_Degradation'] = (df['LapTime (s)'] * df['Cumulative_Degradation']).astype('float32')
    df['_LapTime (s)_*_Cumulative_Degradation_abs'] = (df['LapTime (s)'] * df['Cumulative_Degradation'].abs()).astype('float32')
    df['_LapTime (s)_/_Cumulative_Degradation_abs'] = (df['LapTime (s)'] / (df['Cumulative_Degradation'].abs() + 1e-6)).astype('float32')
    for col in cat_cols:
        if fit:
            codes, uniques = df[col].factorize(); category_map[col] = uniques
        else:
            uniques = category_map[col]; code_map = {c: i for i, c in enumerate(uniques)}
            codes = df[col].map(code_map).fillna(-1).astype('int32')
        df[col] = pd.Series(codes, index=df.index).astype('category')
    for col in num_cols + ['_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber']:
        cat_name = f"{col}_cat_" if col in num_cols else f"{col[1:]}_cat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize(); category_map[col] = uniques
        else:
            uniques = category_map[col]; code_map = {c: i for i, c in enumerate(uniques)}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
        df[cat_name] = pd.Series(codes, index=df.index).astype('category')
    for col in cat_cols + ['Year_cat_', 'PitStop_cat_']:
        count_name = f"_{col}_count" if col in cat_cols else f"_{col[:-1]}_count"
        if fit:
            count_map = df[col].value_counts(); category_map[count_name] = count_map
        else:
            count_map = category_map[count_name]
        df[count_name] = df[col].astype(object).map(count_map).fillna(0).astype('int32')
    bin_config = {'RaceProgress': [200], 'LapTime (s)': [7]}
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            bin_name = f"{col}_{n_bins}_quantile_bin_"
            if fit:
                kb = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile', subsample=None)
                binned = kb.fit_transform(df[[col]]).ravel().astype('int32'); category_map[bin_name] = kb
            else:
                kb = category_map[bin_name]; binned = kb.transform(df[[col]]).ravel().astype('int32')
            df[bin_name] = pd.Series(binned, index=df.index).astype('category')
    combo_names = []
    for cols in important_combos:
        combo_name = '_'.join(cols) + '_'; combo_names.append(combo_name)
        s = df[cols[0]].astype(str)
        for c in cols[1:]: s = s + '_' + df[c].astype(str)
        if fit:
            codes, uniques = pd.factorize(s, sort=False); category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]; code_map = {c: i for i, c in enumerate(uniques)}
            codes = s.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = pd.Series(codes, index=df.index).astype('category')
    new_cat = [c for c in df.columns if c.endswith('_')]
    new_num = [c for c in df.columns if c.startswith('_')]
    return df, new_cat, new_num, combo_names

X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, _, _, _ = feature_engineering(X_test, fit=False)
orig, _, _, _ = feature_engineering(orig, fit=False)
cat_cols_all = cat_cols + new_cat_cols
print('After FE:', X.shape, '| cat:', len(cat_cols_all), '| combos:', combo_names)

# ---------- Model components ----------
class NumericalPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, tfms):
        self._tfms = [t for t in tfms if t in ("median_center","robust_scale","smooth_clip","l2_normalize")]
    def fit(self, X, y=None):
        if "median_center" in self._tfms or "robust_scale" in self._tfms:
            self._median = np.median(X, axis=0)
            q = np.quantile(X,0.75,axis=0) - np.quantile(X,0.25,axis=0)
            z = q == 0.0; q[z] = 0.5*(X.max(axis=0)[z]-X.min(axis=0)[z])
            self._iqr = 1.0/(q+1e-30); self._iqr[q==0.0] = 0.0
        return self
    def transform(self, X, y=None):
        X = X.copy().astype(np.float32)
        for t in self._tfms:
            if t=="median_center": X -= self._median[None,:]
            elif t=="robust_scale": X *= self._iqr[None,:]
            elif t=="smooth_clip": X = X/np.sqrt(1+(X/3)**2)
            elif t=="l2_normalize":
                n = np.linalg.norm(X,axis=1,keepdims=True); X /= np.where(n==0,1.0,n)
        return X

class CategoricalFeatureLayer(nn.Module):
    def __init__(self, n_ens, cat_dims, embed_dim=8, onehot_thresh=8):
        super().__init__()
        self.n_ens=n_ens; self.cat_dims=cat_dims; self.onehot_features=[]
        self.embed_layers=nn.ModuleList(); self._embed_feature_indices=[]
        for i,dim in enumerate(cat_dims):
            if dim<=onehot_thresh: self.onehot_features.append(i)
            else:
                self.embed_layers.append(nn.ModuleList([nn.Embedding(dim,embed_dim) for _ in range(n_ens)]))
                self._embed_feature_indices.append(i)
    def forward(self,x):
        b,k,_=x.shape; feats=[]
        if self.onehot_features:
            ox=x[:,:,self.onehot_features]; od=[self.cat_dims[i] for i in self.onehot_features]
            enc=torch.zeros(b,k,sum(od),device=x.device); st=0
            for idx,dim in enumerate(od):
                enc.scatter_(2, ox[:,:,idx:idx+1].long()+st, 1.0); st+=dim
            feats.append(enc)
        for el,fi in zip(self.embed_layers,self._embed_feature_indices):
            fe=[el[m](x[:,m,fi:fi+1].long()) for m in range(self.n_ens)]
            feats.append(torch.cat(fe,dim=1))
        return torch.cat(feats,dim=2)

class ScalingLayer(nn.Module):
    def __init__(self,n_ens,n_features):
        super().__init__(); self.scale=nn.Parameter(torch.ones(n_ens,n_features))
    def forward(self,x): return x*self.scale[None,:,:]

class NTPLinear(nn.Module):
    def __init__(self,n_ens,in_f,out_f,bias=True):
        super().__init__(); self.in_features=in_f; self.out_features=out_f
        self.weight=nn.Parameter(torch.randn(n_ens,in_f,out_f))
        self.bias=nn.Parameter(torch.randn(n_ens,out_f)) if bias else None
    def forward(self,x):
        x=torch.einsum("bki,kio->bko",x,self.weight)/math.sqrt(self.in_features)
        if self.bias is not None: x=x+self.bias
        return x

class PBLDEmbedding(nn.Module):
    def __init__(self,n_ens,n_features,hidden_dim=16,out_dim=4,freq_scale=0.1,activation=nn.GELU):
        super().__init__(); self.n_ens=n_ens; self.n_features=n_features; self.out_dim=out_dim
        self.w1=nn.Parameter(torch.randn(n_ens,n_features,hidden_dim)*freq_scale)
        self.b1=nn.Parameter(torch.randn(n_ens,n_features,hidden_dim))
        self.w2=nn.Parameter(torch.randn(n_ens,n_features,hidden_dim,out_dim-1)/math.sqrt(hidden_dim))
        self.b2=nn.Parameter(torch.randn(n_ens,n_features,out_dim-1))
        self.act=activation(); nn.init.uniform_(self.b1,-math.pi,math.pi)
    def forward(self,x):
        per=torch.cos(2*math.pi*(x.unsqueeze(-1)*self.w1.unsqueeze(0)+self.b1.unsqueeze(0)))
        tr=self.act(torch.einsum("bkfh,kfhd->bkfd",per,self.w2)+self.b2.unsqueeze(0))
        feat=torch.cat([x.unsqueeze(-1),tr],dim=-1)
        return feat.flatten(start_dim=2)

class RealMLP(nn.Module):
    def __init__(self,output_dim,cat_dims,n_numerical,cfg):
        super().__init__()
        n_ens=cfg["n_ens"]; embed_dim=cfg["embed_dim"]; self.n_ens=n_ens
        self.cate=CategoricalFeatureLayer(n_ens,cat_dims,embed_dim,cfg["onehot_thresh"])
        self.num_embed=PBLDEmbedding(n_ens,n_numerical,cfg["pbld_hidden_dim"],
                                     cfg["pbld_out_dim"],cfg["pbld_freq_scale"],cfg["pbld_activation"])
        num_emb=n_numerical*cfg["pbld_out_dim"]
        cat_emb=sum(c if c<=cfg["onehot_thresh"] else embed_dim for c in cat_dims)
        total=num_emb+cat_emb; act=cfg["activation"]
        layers=[]
        if cfg["add_front_scale"]: layers.append(ScalingLayer(n_ens,total))
        self._dropout_modules=[]; in_dim=total
        for i,h in enumerate(cfg["hidden_dims"]):
            lin=NTPLinear(n_ens,in_dim,h)
            if i==0: self.first_linear=lin
            drop=nn.Dropout(cfg["dropout"]); self._dropout_modules.append(drop)
            layers+=[lin,act(),drop]; in_dim=h
        self.hidden=nn.Sequential(*layers)
        self.output_layer=NTPLinear(n_ens,in_dim,output_dim)
    def forward(self,x_num,x_cat):
        x_num=x_num.unsqueeze(1).expand(-1,self.n_ens,-1)
        x_cat=x_cat.unsqueeze(1).expand(-1,self.n_ens,-1)
        x_num=self.num_embed(x_num); x_cat=self.cate(x_cat)
        x=self.hidden(torch.cat([x_num,x_cat],dim=2))
        return torch.sigmoid(self.output_layer(x))

def apply_schedule(v,p,sched,flat_ratio=0.3):
    if sched=="constant": return v
    if sched=="cos": return v*(math.cos(math.pi*p)+1)/2
    if sched=="flat_cos":
        if p<flat_ratio: return v
        t=(p-flat_ratio)/(1-flat_ratio); return v*(math.cos(math.pi*t)+1)/2
    if sched=="flat_anneal":
        if p<flat_ratio: return v
        t=(p-flat_ratio)/(1-flat_ratio); return v*(1-t)
    if sched=="sqrt_cos": return v*math.sqrt((math.cos(math.pi*p)+1)/2)
    if sched=="expm4t": return v*math.exp(-4*p)
    raise ValueError(sched)

def get_parameter_groups(model,p):
    fid=id(model.first_linear.weight)
    scale_p,pbld_p,first_p,other_p,bias_p=[],[],[],[],[]
    for name,par in model.named_parameters():
        if "num_embed" in name: pbld_p.append(par)
        elif "scale" in name: scale_p.append(par)
        elif id(par)==fid: first_p.append(par)
        elif "bias" in name: bias_p.append(par)
        else: other_p.append(par)
    LR=p["lr"]; WD=p["weight_decay"]
    return [
        {"params":scale_p,"lr":LR*p["lr_scale_mult"],"weight_decay":WD*p["wd_scale_mult"]},
        {"params":pbld_p,"lr":LR*p["pbld_lr_factor"],"weight_decay":WD},
        {"params":first_p,"lr":LR*p["first_layer_lr_factor"],"weight_decay":WD},
        {"params":other_p,"lr":LR,"weight_decay":WD},
        {"params":bias_p,"lr":LR*p["lr_bias_mult"],"weight_decay":WD*p["wd_bias_mult"]},
    ]

def binary_bce_loss(yt,yp,ls=0.0,pos_weight=None,eps=1e-7):
    if ls>0.0: yt=yt*(1.0-ls)+0.5*ls
    yp=yp.clamp(eps,1.0-eps)
    if pos_weight is None:
        loss=-yt*torch.log(yp)-(1.0-yt)*torch.log(1.0-yp)
    else:
        loss=-pos_weight*yt*torch.log(yp)-(1.0-yt)*torch.log(1.0-yp)
    return loss.mean()

class RealMLP_TD_Classifier(BaseEstimator):
    def __init__(self,**kw): self.params={**CONFIG,**kw}
    def fit(self,X_train,y_train,X_val,y_val,cat_col_names=None,ckpt_path="ckpt.pth",X_test=None):
        p=self.params; dev=torch.device(p["device"] if torch.cuda.is_available() else "cpu")
        vb=p["verbosity"]; cat_col_names=cat_col_names or []
        num_col_names=[c for c in X_train.columns if c not in cat_col_names]
        Xtn=X_train[num_col_names].values.astype(np.float32)
        Xvn=X_val[num_col_names].values.astype(np.float32)
        Xtc=X_train[cat_col_names].values.astype(np.int64)
        Xvc=X_val[cat_col_names].values.astype(np.int64)
        y_tr=np.asarray(y_train); y_v=np.asarray(y_val)
        self.preprocessor_=NumericalPreprocessor(p["tfms"]).fit(Xtn)
        Xtn=self.preprocessor_.transform(Xtn); Xvn=self.preprocessor_.transform(Xvn)
        self.cat_col_names_=cat_col_names; self.num_col_names_=num_col_names
        all_cat=[Xtc,Xvc]
        if X_test is not None: all_cat.append(X_test[cat_col_names].values.astype(np.int64))
        cat_dims=(np.concatenate(all_cat,axis=0).max(axis=0)+1).tolist() if cat_col_names else []
        self.cat_dims_=cat_dims
        if cat_dims:
            cmax=np.array(cat_dims)-1; Xtc=np.clip(Xtc,0,cmax); Xvc=np.clip(Xvc,0,cmax)
        classes=np.unique(y_tr); self.classes_=classes
        w=compute_class_weight("balanced",classes=classes,y=y_tr)
        pos_weight=torch.tensor(w[1],dtype=torch.float32,device=dev)
        self.model_=RealMLP(1,cat_dims,Xtn.shape[1],p).to(dev)
        pg=get_parameter_groups(self.model_,p)
        for g in pg: g["lr_base"]=g["lr"]
        opt=torch.optim.AdamW(pg,betas=(p["mom"],p["sq_mom"]))
        Xtn_t=torch.as_tensor(Xtn,dtype=torch.float32,device=dev)
        Xtc_t=torch.as_tensor(Xtc,dtype=torch.long,device=dev)
        ytt=torch.as_tensor(y_tr,dtype=torch.float32,device=dev)
        Xvn_t=torch.as_tensor(Xvn,dtype=torch.float32,device=dev)
        Xvc_t=torch.as_tensor(Xvc,dtype=torch.long,device=dev)
        n_ens=p["n_ens"]; tbs=p["train_bs"]; ebs=p["eval_bs"]; epochs=p["epochs"]
        lrs=p["lr_sched"]; fr=p["flat_ratio"]; total=epochs*len(y_tr)
        order=np.arange(len(y_tr)); best=-np.inf; best_ep=0; best_vp=None
        for epoch in range(epochs):
            self.model_.train()
            for st in range(0,len(y_tr),tbs):
                prog=(epoch*len(y_tr)+st)/total; idx=order[st:st+tbs]
                for g in opt.param_groups: g["lr"]=apply_schedule(g["lr_base"],prog,lrs,fr)
                opt.zero_grad()
                yp=self.model_(Xtn_t[idx],Xtc_t[idx])
                ls=apply_schedule(p["ls_eps"],prog,p["ls_eps_sched"],fr)
                dv=apply_schedule(p["dropout"],prog,p["p_drop_sched"],fr)
                for dm in self.model_._dropout_modules: dm.p=dv
                loss=binary_bce_loss(ytt[idx].repeat_interleave(n_ens),yp.reshape(-1),ls=ls,pos_weight=pos_weight)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model_.parameters(),p["grad_clip"]); opt.step()
            np.random.shuffle(order)
            self.model_.eval()
            with torch.no_grad():
                vp=np.concatenate([self.model_(Xvn_t[s:s+ebs],Xvc_t[s:s+ebs]).mean(dim=1).squeeze(-1).cpu().numpy()
                                   for s in range(0,len(y_v),ebs)],axis=0)
            sc=roc_auc_score(y_v,vp); imp=sc>best
            if imp: best=sc; best_ep=epoch+1; best_vp=vp.copy(); torch.save(self.model_.state_dict(),ckpt_path)
            if vb>=2: print(f"  epoch {epoch+1}/{epochs}  score={sc:.5f}  best={best:.5f}"+(" *" if imp else ""))
        self.model_.load_state_dict(torch.load(ckpt_path))
        self.best_score_=best; self.best_val_probs_=best_vp; self._dev=dev
        if vb>=1: print(f"  -> best: {best:.5f} (epoch {best_ep})")
        return self
    def predict_proba(self,X):
        ebs=self.params["eval_bs"]
        Xn=self.preprocessor_.transform(X[self.num_col_names_].values.astype(np.float32))
        Xc=np.clip(X[self.cat_col_names_].values.astype(np.int64),0,np.array(self.cat_dims_)-1)
        Xn=torch.as_tensor(Xn,dtype=torch.float32,device=self._dev)
        Xc=torch.as_tensor(Xc,dtype=torch.long,device=self._dev)
        self.model_.eval()
        with torch.no_grad():
            pp=np.concatenate([self.model_(Xn[s:s+ebs],Xc[s:s+ebs]).mean(dim=1).squeeze(-1).cpu().numpy()
                               for s in range(0,len(Xn),ebs)],axis=0)
        return np.stack([1.0-pp,pp],axis=1)

# ---------- CONFIG (epochs=8) ----------
CONFIG = {
    "n_ens":16,"embed_dim":6,"onehot_thresh":4,"hidden_dims":[512,64,128],
    "dropout":0.05,"p_drop_sched":"expm4t","activation":nn.SiLU,"add_front_scale":True,
    "pbld_hidden_dim":20,"pbld_out_dim":5,"pbld_freq_scale":5.0,
    "pbld_activation":nn.PReLU,"pbld_lr_factor":0.093,
    "lr":0.008,"mom":0.9,"sq_mom":0.98,"lr_sched":"flat_cos","flat_ratio":0.3,
    "first_layer_lr_factor":1.0,"lr_scale_mult":10.0,"lr_bias_mult":0.1,
    "weight_decay":0.005,"wd_scale_mult":0.1,"wd_bias_mult":0.5,"grad_clip":1.0,
    "ls_eps":0.04,"ls_eps_sched":"cos","tfms":["median_center","robust_scale","smooth_clip"],
    "epochs":8,"train_bs":256,"eval_bs":10240,"verbosity":2,
    "use_early_stopping":False,"early_stopping_additive_patience":10,
    "early_stopping_multiplicative_patience":1,"device":"cuda","random_state":42,
}
print('CONFIG ready. epochs =', CONFIG['epochs'])

# ---------- K-fold training with ORIG-concat + target encoding ----------
FOLDS=5; SEED=42
skf=StratifiedKFold(n_splits=FOLDS,shuffle=True,random_state=SEED)
oof_preds=np.zeros(len(X)); test_preds=np.zeros(len(X_test))
t_all=time.time()
for fold,((tr,va),(otr,_)) in enumerate(zip(skf.split(X,y),skf.split(orig,y_orig)),1):
    X_tr=pd.concat([X.iloc[tr],orig.iloc[otr]],axis=0).reset_index(drop=True)
    y_tr=pd.concat([y.iloc[tr],y_orig.iloc[otr]],axis=0).reset_index(drop=True)
    X_val=X.iloc[va].copy(); y_val=y.iloc[va]; X_tst=X_test.copy()
    te=TargetEncoder(cv=FOLDS,smooth='auto',shuffle=True,random_state=SEED)
    tr_e=te.fit_transform(X_tr[combo_names],y_tr)
    va_e=te.transform(X_val[combo_names]); ts_e=te.transform(X_tst[combo_names])
    ten=[f"_{c}TE" for c in combo_names]
    X_tr[ten]=tr_e; X_val[ten]=va_e; X_tst[ten]=ts_e
    if fold==1: print('Total features:',len(X_tr.columns))
    print('#'*16); print(f'### Fold {fold}/{FOLDS}'); print('#'*16)
    model=RealMLP_TD_Classifier(**CONFIG)
    model.fit(X_tr,y_tr,X_val,y_val,cat_col_names=cat_cols_all,ckpt_path=f'model_fold{fold}.pth')
    oof_preds[va]=model.best_val_probs_
    test_preds+=model.predict_proba(X_tst)[:,1]/FOLDS
    print(f'Fold {fold} | Score: {roc_auc_score(y_val,model.best_val_probs_):.5f}\n')
    torch.cuda.empty_cache()

oof_auc=roc_auc_score(y,oof_preds)
print('='*30); print(f'RealMLP OOF (epochs=8): {oof_auc:.5f}'); print('='*30)
print(f'(epochs=4 was: 0.95356)  | total time: {(time.time()-t_all)/60:.1f} min')

np.save('/kaggle/working/oof_realmlp_e8.npy',oof_preds)
np.save('/kaggle/working/pred_realmlp_e8.npy',test_preds)
sub=pd.DataFrame({ID:test_id,TARGET:test_preds})
sub.to_csv('/kaggle/working/submission.csv',index=False)
print('Saved oof_realmlp_e8.npy, pred_realmlp_e8.npy, submission.csv')

PyTorch: 2.10.0+cu128 | device: cuda
Loaded: (439140, 16) (188165, 15) (101371, 16)
After FE: (439140, 41) | cat: 20 | combos: ['Race_Compound_', 'Race_Year_']
CONFIG ready. epochs = 8
Total features: 43
################
### Fold 1/5
################
  epoch 1/8  score=0.92720  best=0.92720 *
  epoch 2/8  score=0.95232  best=0.95232 *
  epoch 3/8  score=0.95342  best=0.95342 *
  epoch 4/8  score=0.95391  best=0.95391 *
  epoch 5/8  score=0.95435  best=0.95435 *
  epoch 6/8  score=0.95424  best=0.95435
  epoch 7/8  score=0.95383  best=0.95435
  epoch 8/8  score=0.95363  best=0.95435
  -> best: 0.95435 (epoch 5)
Fold 1 | Score: 0.95435

################
### Fold 2/5
################
  epoch 1/8  score=0.92232  best=0.92232 *
  epoch 2/8  score=0.94979  best=0.94979 *
  epoch 3/8  score=0.95163  best=0.95163 *
  epoch 4/8  score=0.95174  best=0.95174 *
  epoch 5/8  score=0.95235  best=0.95235 *
  epoch 6/8  score=0.95168  best=0.95235
  epoch 7/8  score=0.95151  best=0.95235
  epoch 8/8  